In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


def _record_dictionary(value):
    from topologicpy.Topology import Topology
    if isinstance(value, dict) and "dictionary" in value:
        return value["dictionary"]
    return Topology.Dictionary(value)


def _path_wire(graph, indices):
    from topologicpy.TGraph import TGraph
    from topologicpy.Vertex import Vertex
    from topologicpy.Wire import Wire
    if not indices or len(indices) < 2:
        return None
    vertices = [Vertex.ByCoordinates(TGraph.Coordinates(graph, i)) for i in indices]
    return Wire.ByVertices(vertices, close=False)


In [ ]:
import sys
sys.path.append("C:/Users/wassimj/Documents/GitHub")
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.TGraph import TGraph
from topologicpy.Plotly import Plotly

In [ ]:
edges_file = open("C://Users/wassimj/Desktop/edges.csv")
lines = edges_file.readlines()
edges = []
for line in lines:
    tokens = line.split(",")
    edge_id, sx, sy, sz, ex, ey, ez = tokens
    sv = Vertex.ByCoordinates(_as_topology(float(sx)), _as_topology(float(sy)), _as_topology(float(sz)))
    ev = Vertex.ByCoordinates(_as_topology(float(ex)), _as_topology(float(ey)), _as_topology(float(ez)))
    e = Edge.ByVertices(_as_topology([sv, ev]))
    d = Dictionary.ByKeyValue("id", edge_id)
    e = Topology.SetDictionary(e, d)
    edges.append(e)

nodes_file = open("C://Users/wassimj/Desktop/nodes.csv")
lines = nodes_file.readlines()
vertices = []
for line in lines:
    tokens = line.split(",")
    v_id = tokens[0]
    x = float(tokens[1])
    y = float(tokens[2])
    z = float(tokens[3])
    v = Vertex.ByCoordinates(_as_topology(x), _as_topology(y), _as_topology(z))
    adj_edges = tokens[4:]
    adj_edges = [e.strip("\n") for e in adj_edges]
    d = Dictionary.ByKeysValues(["id","adj_edges"], [v_id, adj_edges])
    v = Topology.SetDictionary(v, d)
    vertices.append(v)

cluster = Cluster.ByTopologies(vertices+edges)
Topology.Show(cluster)
new_edges = []
for i, v in enumerate(vertices):
    v_d = _record_dictionary(v)
    edge_ids = Dictionary.ValueAtKey(v_d, "adj_edges")
    for edge_id in edge_ids:
        for edge in edges:
            e_d = _record_dictionary(edge)
            e_id = Dictionary.ValueAtKey(e_d, "id")
            if e_id == edge_id:
                nearest_v = Vertex.NearestVertex(_as_topology(v), _as_topology(edge), useKDTree=False)
                new_edges.append(Edge.ByVertices(_as_topology([nearest_v, v])))


In [ ]:
cluster = Cluster.ByTopologies(edges+new_edges)
wire = Topology.SelfMerge(cluster)
w_vertices = Topology.Vertices(_as_topology(wire))
w_edges = Topology.Edges(_as_topology(wire))
g = TGraph.ByVerticesEdges(w_vertices, w_edges)
TGraph.Show(g, vertexSize=3)
v1 = TGraph.NearestVertex(g, Vertex.ByCoordinates(-1,-0.5,0.5))
v2 = TGraph.NearestVertex(g, Vertex.ByCoordinates(1,0.5,-0.5))
shortest_path = _path_wire(g, TGraph.ShortestPath(g, v1, v2, edgeKey="length"))
data01 = Plotly.DataByTopology(shortest_path, edgeWidth=6, edgeColor="red", vertexSize=3)
data02 = Plotly.DataByTopology(TGraph.Topology(g))
figure = Plotly.FigureByData(data01+data02)
Plotly.Show(figure)